In [1]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score

/home/franciele-santos-da-silva/Documentos/TDE-NPL/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../data/anime_reviews.csv")

# print(df.head())
# display("Total de dados:", len(df))
display(df)

,review_id,anime_mal_id,anime_title,username,score,tags,review_text,date,episodes_watched,is_spoiler,is_preliminary,reactions_overall,reactions_nice,reactions_love_it,reactions_funny,reactions_confusing,reactions_informative,reactions_well_written,reactions_creative
0,602806,16782.0,Kotonoha no Niwa,gimmefood365,8,Recommended,Makoto Shinkai's The Garden of Words is one of...,2026-04-26,NaN,False,False,0,0,0,0,0,0,0,0
1,602801,50594.0,Suzume no Tojimari,Andrew2706,10,Recommended,Y con es termino de ver todas las peliculas de...,2026-04-26,NaN,False,False,0,0,0,0,0,0,0,0
2,602799,5680.0,K-On!,yui_keion_fan,10,Recommended,My honest opinion on K-ON!: I absolutely LOVE ...,2026-04-26,NaN,False,False,0,0,0,0,0,0,0,0
3,602793,1735.0,Naruto: Shippuuden,Kiunax3,10,Recommended,"Uh, I don't even even know where to start. Hon...",2026-04-26,NaN,False,False,0,0,0,0,0,0,0,0
4,602787,227.0,FLCL,CR7SPY,8,Recommended,"""FLCL"" both walks and works only on a thin lin...",2026-04-26,NaN,False,False,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1485,314523,NaN,NaN,Honghan1995,10,Recommended,The third season of the My Hero Academia anime...,2019-08-03,NaN,False,False,6,5,0,0,1,0,0,0
1486,477616,NaN,NaN,FAKECROSS97,8,Recommended,Part 1 English Part 2 Bahasa Indonesia -------...,2023-03-26,NaN,False,False,1,0,0,0,0,0,0,1
1487,595806,NaN,NaN,Lost_crimes,8,Recommended,My Hero Academia season 3 is where things fina...,2026-03-06,NaN,False,False,2,2,0,0,0,0,0,0
1488,369620,NaN,NaN,Cuteanimeboys,7,Recommended,*CONTAIN SPOILER* Story - overall it was very ...,2020-12-24,NaN,False,False,2,2,0,0,0,0,0,0


In [3]:
df = df.dropna(subset=['review_text', 'score'])

df['label'] = df['score'].apply(lambda x: 1 if x >= 7 else 0)

# evita erro de sample
df = df.sample(min(2000, len(df)))

df = df[['review_text', 'label']]

df.head()

,review_text,label
943,"A series that has filler, emotions and one of ...",1
1109,Koe no Katachi is a brilliantly crafted tale o...,1
761,"Seeing this movie so high, at the top ratings,...",1
256,One-Punch Man continuously combines the humour...,1
448,"First, let me state that I watched Sword Art O...",0


In [4]:
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2)

In [5]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(example):
    return tokenizer(example['review_text'], truncation=True, padding='max_length')

dataset = dataset.map(tokenize, batched=True)

Map: 100%|██████████| 298/298 [00:00<00:00, 5305.80 examples/s]


In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 13498.66it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
training_args = TrainingArguments(
    output_dir="../models/",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_dir="../logs",
    save_strategy="no"
)

trainer = Trainer(
    model=model, 
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test']
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss


TrainOutput(global_step=149, training_loss=0.6126713080694212, metrics={'train_runtime': 169.3131, 'train_samples_per_second': 7.04, 'train_steps_per_second': 0.88, 'total_flos': 157901139197952.0, 'train_loss': 0.6126713080694212, 'epoch': 1.0})

In [8]:
preds = trainer.predict(dataset['test'])

y_pred = preds.predictions.argmax(axis=1)
y_true = preds.label_ids

acc = accuracy_score(y_true, y_pred)
print(f"Acurácia: {acc:.2f}")

Acurácia: 0.78
